# BirdCLEF+ 2026 - exp002 Training

**Kaggle Notebook (GPU) で実行する学習ノートブック**

完了後、Output の `best_fold0.pth` を Kaggle Dataset `birdclef2026-exp002-weights` として保存してください。

### exp001 からの主な改善点

| 改善点 | exp001 | exp002 |
|--------|--------|--------|
| アーキテクチャ | EfficientNet-B0 + GlobalAvgPool | SED + AttBlockV2 attention pooling |
| 損失関数 | BCEWithLogitsLoss | CrossEntropyLoss（主ラベルのみ） |
| 推論時活性化 | sigmoid | sigmoid（CE訓練→sigmoid推論） |
| 入力長 | 5秒 | 10秒 |
| クラス不均衡 | なし | WeightedRandomSampler (w=(count/N)^-0.5) |
| SpecAugment | なし | FrequencyMasking + TimeMasking |
| Mel変換 | librosa (CPU) | torchaudio (GPU) |

In [2]:
!pip install -q timm torchaudio scikit-learn

In [ ]:
import os, pathlib, glob

# ── パス設定 ──────────────────────────────────────────────────
_s = glob.glob('/kaggle/input/**/sample_submission.csv', recursive=True)
COMP_DIR        = os.path.dirname(_s[0]) if _s else '/kaggle/input/birdclef-2026'
TRAIN_CSV       = f'{COMP_DIR}/train.csv'
TAXONOMY_CSV    = f'{COMP_DIR}/taxonomy.csv'
SAMPLE_SUB_CSV  = f'{COMP_DIR}/sample_submission.csv'
TRAIN_AUDIO_DIR = f'{COMP_DIR}/train_audio'

# 重みの出力先（Kaggle Notebook は /kaggle/working/ が Output になる）
WEIGHT_DIR = '/kaggle/working/weights'
LOG_DIR    = '/kaggle/working/logs'

os.makedirs(WEIGHT_DIR, exist_ok=True)
os.makedirs(LOG_DIR,    exist_ok=True)

for label, path in [('TRAIN_CSV', TRAIN_CSV), ('SAMPLE_SUB_CSV', SAMPLE_SUB_CSV), ('TRAIN_AUDIO_DIR', TRAIN_AUDIO_DIR)]:
    print(f'  {"OK" if pathlib.Path(path).exists() else "NG"} {label}: {path}')

In [ ]:
# ── ハイパーパラメータ ────────────────────────────────────────
CFG = dict(
    # 音声
    sample_rate      = 32000,
    n_samples        = 32000 * 10,   # 10秒（exp001は5秒）
    n_mels           = 128,
    n_fft            = 1024,
    hop_length       = 320,
    fmin             = 20,
    fmax             = 16000,
    # 学習
    seed             = 42,
    n_folds          = 5,
    train_fold       = 0,
    epochs           = 20,
    batch_size       = 32,
    num_workers      = 2,
    lr               = 1e-3,
    weight_decay     = 1e-4,
    warmup_epochs    = 1,
    use_amp          = False,   # NaN対策のためFalse（T4でも十分速い）
    label_smoothing  = 0.05,
    # モデル
    model_name       = 'tf_efficientnet_b0_ns',
    num_classes      = 234,
    # SpecAugment
    spec_aug         = True,
    freq_mask_param  = 27,       # 128mel の ~20%
    time_mask_param  = 100,      # 1000frame の ~10%
    # Gaussian Noise
    gaussian_noise   = True,
    noise_std        = 0.005,
    noise_prob       = 0.5,
)

In [6]:
import ast, random, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
from torch.cuda.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
import timm
from tqdm.notebook import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'torch     : {torch.__version__}')
print(f'torchaudio: {torchaudio.__version__}')
print(f'device    : {DEVICE}')

torch     : 2.10.0+cu128
torchaudio: 2.10.0+cu128
device    : cuda


In [7]:
# ── ユーティリティ ────────────────────────────────────────────
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

class AverageMeter:
    def __init__(self):
        self.reset()
    def reset(self):
        self.val = self.avg = self.sum = self.count = 0.0
    def update(self, val, n=1):
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

def compute_roc_auc(targets, preds, labels):
    aucs = []
    for i in range(len(labels)):
        if targets[:, i].sum() == 0:
            continue
        try:
            aucs.append(roc_auc_score(targets[:, i], preds[:, i]))
        except Exception:
            pass
    return float(np.mean(aucs)) if aucs else 0.0

set_seed(CFG['seed'])

In [8]:
# ── GPU上のMel変換・SpecAugment ───────────────────────────────
mel_transform = nn.Sequential(
    T.MelSpectrogram(
        sample_rate = CFG['sample_rate'],
        n_fft       = CFG['n_fft'],
        hop_length  = CFG['hop_length'],
        n_mels      = CFG['n_mels'],
        f_min       = CFG['fmin'],
        f_max       = CFG['fmax'],
    ),
    T.AmplitudeToDB(top_db=80),
).to(DEVICE)

freq_masking = T.FrequencyMasking(freq_mask_param=CFG['freq_mask_param']).to(DEVICE)
time_masking = T.TimeMasking(time_mask_param=CFG['time_mask_param']).to(DEVICE)

def waveform_to_spec(waveforms: torch.Tensor, training: bool = False) -> torch.Tensor:
    """(B, n_samples) → (B, 1, n_mels, T)  ※GPU上で実行"""
    with torch.no_grad():
        specs = mel_transform(waveforms)          # (B, n_mels, T)
    # 正規化: [0, 1]
    specs = specs - specs.amin(dim=(-2, -1), keepdim=True)
    specs = specs / (specs.amax(dim=(-2, -1), keepdim=True) + 1e-8)
    # SpecAugment (訓練時のみ)
    if training and CFG['spec_aug']:
        specs = freq_masking(specs)               # FrequencyMasking
        specs = time_masking(specs)               # TimeMasking
    return specs.unsqueeze(1)                     # (B, 1, n_mels, T)

print(f'Mel output shape (10s): ({CFG["n_mels"]}, {CFG["n_samples"] // CFG["hop_length"] + 1})')

Mel output shape (10s): (128, 1001)


In [9]:
# ── Dataset（waveformを返す・mel変換はGPU上で行う）──────────────
class BirdCLEFDataset(Dataset):
    def __init__(self, df, label2idx, mode='train'):
        self.df       = df.reset_index(drop=True)
        self.label2idx = label2idx
        self.mode     = mode
        self.n        = CFG['n_samples']

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        audio = self._load_audio(row['filename'])
        primary_idx = self.label2idx.get(row['primary_label'], 0)
        return audio, primary_idx

    def _load_audio(self, filename):
        path = f'{TRAIN_AUDIO_DIR}/{filename}'
        try:
            waveform, sr = torchaudio.load(path)
        except Exception:
            return torch.zeros(self.n)
        if sr != CFG['sample_rate']:
            waveform = torchaudio.functional.resample(waveform, sr, CFG['sample_rate'])
        audio = waveform.mean(dim=0)  # モノラル化 → (L,)

        # クロップ or パディング
        if len(audio) >= self.n:
            start = random.randint(0, len(audio) - self.n) if self.mode == 'train' else (len(audio) - self.n) // 2
            audio = audio[start:start + self.n]
        else:
            audio = F.pad(audio, (0, self.n - len(audio)))

        # Gaussian Noise
        if self.mode == 'train' and CFG['gaussian_noise'] and random.random() < CFG['noise_prob']:
            audio = audio + torch.randn_like(audio) * CFG['noise_std']

        return audio  # (n_samples,)

In [10]:
# ── SED モデル（AttBlockV2 + EfficientNet-B0）────────────────
class AttBlockV2(nn.Module):
    """時間方向のAttention Pooling
    各クラスが「どの時刻フレームを重視するか」を学習する。
    input:  (B, C, T)
    output: (B, num_classes)
    """
    def __init__(self, in_features: int, num_classes: int):
        super().__init__()
        self.att = nn.Conv1d(in_features, num_classes, kernel_size=1, bias=True)
        self.cla = nn.Conv1d(in_features, num_classes, kernel_size=1, bias=True)
        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.att.weight)
        nn.init.xavier_uniform_(self.cla.weight)
        nn.init.constant_(self.att.bias, 0)
        nn.init.constant_(self.cla.bias, 0)

    def forward(self, x):
        # x: (B, C, T)
        att = torch.softmax(torch.tanh(self.att(x)), dim=-1)  # (B, num_classes, T)
        cla = self.cla(x)                                      # (B, num_classes, T)  logits
        clip_logit = (att * cla).sum(dim=-1)                   # (B, num_classes)
        return clip_logit


class BirdCLEFSED(nn.Module):
    """Sound Event Detection モデル
    EfficientNet-B0 のバックボーンから空間特徴マップを取り出し、
    周波数次元を平均プールして時系列にし、AttBlockV2 で集約する。
    """
    def __init__(self, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(
            CFG['model_name'],
            pretrained=pretrained,
            in_chans=1,
            num_classes=0,
            global_pool='',    # 空間プールを無効化 → (B, C, H, W) を返す
        )
        in_features = self.backbone.num_features  # EfficientNet-B0: 1280
        self.bn        = nn.BatchNorm1d(in_features)
        self.dropout   = nn.Dropout(p=0.3)
        self.att_block = AttBlockV2(in_features, CFG['num_classes'])

    def forward(self, x):
        # x: (B, 1, n_mels, T_frames)
        feat = self.backbone.forward_features(x)   # (B, 1280, H, W)
        feat = feat.mean(dim=2)                    # 周波数次元を平均 → (B, 1280, W)
        feat = self.bn(feat)                       # BatchNorm1d
        feat = self.dropout(feat)
        clip_logit = self.att_block(feat)          # (B, 234)
        return clip_logit

In [ ]:
# ── データ読み込み・fold分割 ───────────────────────────────────
train_df = pd.read_csv(TRAIN_CSV)
sub_df   = pd.read_csv(SAMPLE_SUB_CSV, nrows=0)
LABELS   = [c for c in sub_df.columns if c != 'row_id']
LABEL2IDX = {l: i for i, l in enumerate(LABELS)}

skf = StratifiedKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=CFG['seed'])
train_df['fold'] = -1
for fold, (_, val_idx) in enumerate(skf.split(train_df, train_df['primary_label'])):
    train_df.loc[val_idx, 'fold'] = fold

FOLD  = CFG['train_fold']
tr_df = train_df[train_df['fold'] != FOLD].reset_index(drop=True)
va_df = train_df[train_df['fold'] == FOLD].reset_index(drop=True)

# ── サンプル数の制限（動作確認用）────────────────────────────
# 全データで学習する場合は MAX_SAMPLES = None にする
MAX_SAMPLES = None
if MAX_SAMPLES:
    tr_df = tr_df.sample(MAX_SAMPLES, random_state=CFG['seed']).reset_index(drop=True)
    va_df = va_df.sample(MAX_SAMPLES, random_state=CFG['seed']).reset_index(drop=True)

print(f'Fold {FOLD}: train={len(tr_df)}, valid={len(va_df)}, classes={len(LABELS)}')
print(f'Primary label stats: min={train_df["primary_label"].value_counts().min()}, '
      f'max={train_df["primary_label"].value_counts().max()}, '
      f'mean={train_df["primary_label"].value_counts().mean():.1f}')

In [12]:
# ── WeightedRandomSampler（クラス不均衡対策）──────────────────
# weight = (count / total) ^ (-0.5)  ← 希少クラスを多くサンプリング
counts       = tr_df['primary_label'].value_counts()
total        = len(tr_df)
sample_weights = tr_df['primary_label'].map(
    lambda x: (counts.get(x, 1) / total) ** (-0.5)
).values
sampler = WeightedRandomSampler(
    weights     = torch.tensor(sample_weights, dtype=torch.float64),
    num_samples = len(sample_weights),
    replacement = True,
)
print(f'WeightedRandomSampler: min_weight={sample_weights.min():.3f}, max_weight={sample_weights.max():.3f}')

# ── DataLoader ────────────────────────────────────────────────
tr_loader = DataLoader(
    BirdCLEFDataset(tr_df, LABEL2IDX, mode='train'),
    batch_size  = CFG['batch_size'],
    sampler     = sampler,               # shuffle=True の代わりに sampler を使用
    num_workers = CFG['num_workers'],
    pin_memory  = True,
    drop_last   = True,
)
va_loader = DataLoader(
    BirdCLEFDataset(va_df, LABEL2IDX, mode='valid'),
    batch_size  = CFG['batch_size'] * 2,
    shuffle     = False,
    num_workers = CFG['num_workers'],
    pin_memory  = True,
)
print(f'train batches: {len(tr_loader)}, valid batches: {len(va_loader)}')

WeightedRandomSampler: min_weight=8.442, max_weight=168.639
train batches: 888, valid batches: 112


In [13]:
# ── 学習・評価ループ ──────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, scaler):
    model.train()
    loss_meter = AverageMeter()

    for waveforms, targets in tqdm(loader, desc='  train', leave=False):
        waveforms = waveforms.to(DEVICE)  # (B, n_samples)
        targets   = targets.to(DEVICE)    # (B,) long

        optimizer.zero_grad()
        with autocast(enabled=CFG['use_amp']):
            specs  = waveform_to_spec(waveforms, training=True)  # (B, 1, n_mels, T)
            logits = model(specs)                                  # (B, num_classes)
            # CrossEntropyLoss: 主ラベルのクラスインデックスで学習
            loss   = F.cross_entropy(logits, targets, label_smoothing=CFG['label_smoothing'])

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        loss_meter.update(loss.item(), waveforms.size(0))

    return loss_meter.avg


@torch.no_grad()
def validate(model, loader):
    model.eval()
    loss_meter = AverageMeter()
    all_preds, all_targets = [], []

    for waveforms, targets in tqdm(loader, desc='  valid', leave=False):
        waveforms = waveforms.to(DEVICE)
        targets   = targets.to(DEVICE)

        with autocast(enabled=CFG['use_amp']):
            specs  = waveform_to_spec(waveforms, training=False)
            logits = model(specs)
            loss   = F.cross_entropy(logits, targets, label_smoothing=CFG['label_smoothing'])

        loss_meter.update(loss.item(), waveforms.size(0))

        # 推論時は sigmoid でクラスごとの確率を得る（CEで訓練 → sigmoid推論）
        probs = torch.sigmoid(logits).cpu().numpy()  # (B, num_classes)
        all_preds.append(probs)

        # ROC AUC 計算用: one-hot に変換
        one_hot = np.zeros((len(targets), len(LABELS)), dtype=np.float32)
        for i, t in enumerate(targets.cpu().numpy()):
            one_hot[i, t] = 1.0
        all_targets.append(one_hot)

    all_preds   = np.concatenate(all_preds,   axis=0)
    all_targets = np.concatenate(all_targets, axis=0)
    auc = compute_roc_auc(all_targets, all_preds, LABELS)
    return loss_meter.avg, auc

In [ ]:
# ── 学習実行 ──────────────────────────────────────────────────
model     = BirdCLEFSED(pretrained=True).to(DEVICE)
optimizer = AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scheduler = CosineAnnealingLR(optimizer, T_max=CFG['epochs'], eta_min=1e-6)
scaler    = GradScaler(enabled=CFG['use_amp'])

WEIGHT_PATH  = f'{WEIGHT_DIR}/best_fold{FOLD}.pth'
CKPT_PATH    = f'{WEIGHT_DIR}/checkpoint_fold{FOLD}.pth'  # 定期チェックポイント
best_auc, best_epoch = 0.0, 0
start_epoch  = 1
log_rows     = []

# ── チェックポイントからの再開 ────────────────────────────────
if os.path.exists(CKPT_PATH):
    print(f'Resuming from checkpoint: {CKPT_PATH}')
    ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    scaler.load_state_dict(ckpt['scaler_state_dict'])
    start_epoch = ckpt['epoch'] + 1
    best_auc    = ckpt['best_auc']
    log_rows    = ckpt.get('log_rows', [])
    print(f'  Resumed from epoch {ckpt["epoch"]} | best AUC so far: {best_auc:.4f}')
else:
    print('No checkpoint found. Starting from scratch.')

print('=' * 60)
print(f'Training exp002 | Fold {FOLD} | epoch {start_epoch}~{CFG["epochs"]} | {DEVICE}')
print(f'Model: SED + AttBlockV2 + {CFG["model_name"]}')
print(f'Input: {CFG["n_samples"]/CFG["sample_rate"]:.0f}s | Loss: CrossEntropyLoss')
print('=' * 60)

for epoch in range(start_epoch, CFG['epochs'] + 1):
    # Warmup
    if epoch <= CFG['warmup_epochs']:
        for pg in optimizer.param_groups:
            pg['lr'] = CFG['lr'] * epoch / CFG['warmup_epochs']

    tr_loss = train_one_epoch(model, tr_loader, optimizer, scaler)

    if epoch > CFG['warmup_epochs']:
        scheduler.step()

    va_loss, va_auc = validate(model, va_loader)
    lr = optimizer.param_groups[0]['lr']
    log_rows.append(dict(epoch=epoch, lr=lr, tr_loss=tr_loss, va_loss=va_loss, va_auc=va_auc))

    is_best = va_auc > best_auc
    print(f'Epoch {epoch:03d}/{CFG["epochs"]}'
          f' | LR={lr:.2e} | Train={tr_loss:.4f} | Valid={va_loss:.4f} | AUC={va_auc:.4f}'
          f'{" ← best" if is_best else ""}')

    # ベストモデルの保存
    if is_best:
        best_auc, best_epoch = va_auc, epoch
        torch.save({
            'epoch':            epoch,
            'model_state_dict': model.state_dict(),
            'best_auc':         best_auc,
            'labels':           LABELS,
            'cfg':              CFG,
        }, WEIGHT_PATH)
        print(f'  Best saved: {WEIGHT_PATH}')

    # 定期チェックポイント（5epochごと）: セッション切れ対策
    if epoch % 5 == 0:
        torch.save({
            'epoch':                epoch,
            'model_state_dict':     model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict':    scaler.state_dict(),
            'best_auc':             best_auc,
            'labels':               LABELS,
            'cfg':                  CFG,
            'log_rows':             log_rows,
        }, CKPT_PATH)
        print(f'  Checkpoint saved (epoch {epoch}): {CKPT_PATH}')

    pd.DataFrame(log_rows).to_csv(f'{LOG_DIR}/train_log_fold{FOLD}.csv', index=False)

print('=' * 60)
print(f'Best AUC: {best_auc:.4f} @ Epoch {best_epoch}')
print('=' * 60)

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

No checkpoint found. Starting from scratch.
Training exp002 | Fold 0 | epoch 1~20 | cuda
Model: SED + AttBlockV2 + tf_efficientnet_b0_ns
Input: 10s | Loss: CrossEntropyLoss


  train:   0%|          | 0/888 [00:00<?, ?it/s]

In [ ]:
# ── 学習曲線の確認 ────────────────────────────────────────────
import matplotlib.pyplot as plt

log_df = pd.DataFrame(log_rows)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, title in zip(axes, ['tr_loss', 'va_loss', 'va_auc'], ['Train Loss', 'Valid Loss', 'Valid AUC']):
    ax.plot(log_df['epoch'], log_df[col])
    ax.set_title(title)
    ax.set_xlabel('Epoch')
plt.tight_layout()
plt.savefig(f'{LOG_DIR}/train_curve_fold{FOLD}.png', dpi=100)
plt.show()
print(f'Weight saved to: {WEIGHT_PATH}')

In [ ]:
# ── 学習完了後の手順 ──────────────────────────────────────────
# Kaggle Notebook の Output（/kaggle/working/weights/）に以下が保存されています:
#   - best_fold0.pth        … AUC最良モデル（提出用）
#   - checkpoint_fold0.pth  … 最新チェックポイント（再開用）
#
# 【Kaggle Dataset への保存手順】
# 1. ノートブック右上「Save Version」→「Save & Run All」でコミット実行
# 2. 実行完了後、Output タブ →「weights/best_fold0.pth」を確認
# 3. Output右上「...」→「New Dataset」→ Dataset名: birdclef2026-exp002-weights で保存
# 4. submission.ipynb の Input にそのDatasetを追加して提出

print('Training complete!')
print(f'Best model : {WEIGHT_DIR}/best_fold0.pth')
print(f'Checkpoint : {WEIGHT_DIR}/checkpoint_fold0.pth')
print()
print('次のステップ: Output → weights/best_fold0.pth を Kaggle Dataset として保存')